In [1]:
%pip install faker tqdm

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 20.9 MB/s  0:00:00

   ---------------------------------------- 0/2 [tqdm]
   ---------------------------------------- 0/2 [tqdm]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   -------------------- ------------------- 1/2 [faker]
   ---------------

In [2]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta
from tqdm import tqdm

fake = Faker()

OUTPUT_DIR = "D:\\Riverstone Restaurant Group\\Datasets\\"

# scale settings
NUM_RESTAURANTS = 210
NUM_CUSTOMERS = 150000
NUM_EMPLOYEES = 5200
NUM_MENU_ITEMS = 96
NUM_INGREDIENTS = 110
NUM_ORDERS = 250000
NUM_SHIFT_ROWS = 100000
NUM_INVENTORY_TXN = 200000

START_DATE = datetime(2025,1,1)
END_DATE = datetime(2025,12,31)

# ------------------------
# concepts
# ------------------------

concepts = pd.DataFrame([
    [1,"Riverstone Grill","full_service","high",26,18],
    [2,"Iron Skillet Kitchen","fast_casual","mid",13,7],
    [3,"Jacks Burger Co","quick_service","low",9,4]
], columns=[
"concept_id",
"concept_name",
"service_model",
"price_tier",
"avg_ticket_target",
"avg_prep_minutes_target"
])

concepts.to_csv(OUTPUT_DIR+"concepts.csv",index=False)

# ------------------------
# restaurants
# ------------------------

states = ["TX","TN","KY","IN","OH","GA","FL","NC","MO","OK","AZ","CO"]

restaurants = []

for i in range(NUM_RESTAURANTS):

    concept = random.choice([1,2,3])

    restaurants.append([
        i+1,
        concept,
        f"Store {i+1}",
        i+1000,
        fake.city(),
        random.choice(states),
        random.choice(["Midwest","South","Southeast"]),
        fake.date_between(start_date="-10y",end_date="-1y"),
        random.randint(60,300),
        random.choice([0,1]),
        random.uniform(30,45),
        random.uniform(-110,-80),
        random.choice(["urban","suburban","rural"]),
        "active"
    ])

restaurants = pd.DataFrame(restaurants,columns=[
"restaurant_id","concept_id","restaurant_name","store_number",
"city","state","region","open_date","seating_capacity",
"drive_thru_flag","latitude","longitude","market_type","status"
])

restaurants.to_csv(OUTPUT_DIR+"restaurants.csv",index=False)

# ------------------------
# menu categories
# ------------------------

categories = []

cat_names = [
"Appetizers","Steaks","Burgers","Sandwiches",
"Sides","Desserts","Beverages","Bowls"
]

id_counter = 1

for concept in [1,2,3]:
    for name in cat_names:
        categories.append([id_counter,concept,name,id_counter])
        id_counter += 1

menu_categories = pd.DataFrame(categories,columns=[
"menu_category_id","concept_id","menu_category_name","display_order"
])

menu_categories.to_csv(OUTPUT_DIR+"menu_categories.csv",index=False)

# ------------------------
# menu items
# ------------------------

menu_items = []

for i in range(NUM_MENU_ITEMS):

    concept = random.choice([1,2,3])
    category = random.choice(menu_categories.menu_category_id.tolist())

    price = round(random.uniform(5,30),2)

    menu_items.append([
        i+1,
        concept,
        category,
        fake.word().capitalize()+" Item",
        random.choice(["food","beverage"]),
        price,
        round(price*random.uniform(.25,.45),2),
        random.randint(3,20),
        1,
        fake.date_between(start_date="-5y",end_date="-1y")
    ])

menu_items = pd.DataFrame(menu_items,columns=[
"menu_item_id","concept_id","menu_category_id","item_name",
"item_type","base_price","standard_food_cost",
"standard_prep_minutes","is_active","launch_date"
])

menu_items.to_csv(OUTPUT_DIR+"menu_items.csv",index=False)

# ------------------------
# ingredients
# ------------------------

ingredients = []

for i in range(NUM_INGREDIENTS):

    ingredients.append([
        i+1,
        fake.word(),
        random.choice(["protein","dairy","produce","dry"]),
        random.choice(["lbs","oz","each"]),
        round(random.uniform(.2,5),2),
        random.randint(3,20),
        random.choice([0,1])
    ])

ingredients = pd.DataFrame(ingredients,columns=[
"ingredient_id","ingredient_name","ingredient_category",
"unit_of_measure","standard_unit_cost","shelf_life_days","is_perishable"
])

ingredients.to_csv(OUTPUT_DIR+"ingredients.csv",index=False)

# ------------------------
# menu item ingredients
# ------------------------

recipes = []

for i in range(500):

    recipes.append([
        random.randint(1,NUM_MENU_ITEMS),
        random.randint(1,NUM_INGREDIENTS),
        round(random.uniform(.1,2),2),
        round(random.uniform(.01,.05),3)
    ])

menu_item_ingredients = pd.DataFrame(recipes,columns=[
"menu_item_id","ingredient_id","quantity_per_item","waste_factor_pct"
])

menu_item_ingredients.to_csv(OUTPUT_DIR+"menu_item_ingredients.csv",index=False)

# ------------------------
# customers
# ------------------------

customers = []

for i in tqdm(range(NUM_CUSTOMERS)):

    customers.append([
        i+1,
        fake.city(),
        random.choice(states),
        fake.date_between(start_date="-3y",end_date="today"),
        random.choice([0,1]),
        random.choice(["18-25","26-35","36-50","50+"]),
        random.choice(["in_store","mobile","delivery"])
    ])

customers = pd.DataFrame(customers,columns=[
"customer_id","customer_city","customer_state",
"signup_date","loyalty_member_flag","birth_year_band",
"preferred_order_channel"
])

customers.to_csv(OUTPUT_DIR+"customers.csv",index=False)

# ------------------------
# employees
# ------------------------

employees = []

for i in range(NUM_EMPLOYEES):

    restaurant = random.randint(1,NUM_RESTAURANTS)

    employees.append([
        i+1,
        restaurant,
        restaurants.loc[restaurants.restaurant_id==restaurant,"concept_id"].values[0],
        random.choice(["server","cook","manager","cashier"]),
        fake.date_between(start_date="-5y",end_date="-30d"),
        "active",
        round(random.uniform(10,30),2),
        random.choice([0,1]),
        random.choice([0,1])
    ])

employees = pd.DataFrame(employees,columns=[
"employee_id","restaurant_id","concept_id","job_title",
"hire_date","employment_status","hourly_rate",
"full_time_flag","manager_flag"
])

employees.to_csv(OUTPUT_DIR+"employees.csv",index=False)

# ------------------------
# orders
# ------------------------

orders = []

date_range = (END_DATE-START_DATE).days

for i in tqdm(range(NUM_ORDERS)):

    restaurant = random.randint(1,NUM_RESTAURANTS)
    concept = restaurants.loc[
        restaurants.restaurant_id==restaurant,
        "concept_id"
    ].values[0]

    order_time = START_DATE + timedelta(days=random.randint(0,date_range))

    subtotal = round(random.uniform(10,60),2)

    orders.append([
        i+1,
        restaurant,
        concept,
        random.randint(1,NUM_CUSTOMERS),
        order_time,
        random.choice(["in_store","mobile","delivery"]),
        random.randint(1,6),
        subtotal,
        0,
        round(subtotal*.08,2),
        round(subtotal*1.08,2),
        random.choice(["card","cash"]),
        order_time,
        order_time+timedelta(minutes=random.randint(5,20)),
        random.choice(["dine_in","takeout","drive_thru"])
    ])

orders = pd.DataFrame(orders,columns=[
"order_id","restaurant_id","concept_id","customer_id",
"order_datetime","order_channel","party_size","subtotal",
"discount_amount","tax_amount","total_amount","payment_type",
"prep_start_datetime","order_ready_datetime","service_mode"
])

orders.to_csv(OUTPUT_DIR+"orders.csv",index=False)

# ------------------------
# order items
# ------------------------

items = []

order_item_id = 1

for order in tqdm(orders.order_id):

    for _ in range(random.randint(1,4)):

        menu = random.randint(1,NUM_MENU_ITEMS)

        qty = random.randint(1,3)

        price = menu_items.loc[
            menu_items.menu_item_id==menu,
            "base_price"
        ].values[0]

        items.append([
            order_item_id,
            order,
            menu,
            qty,
            price,
            round(qty*price,2)
        ])

        order_item_id+=1

order_items = pd.DataFrame(items,columns=[
"order_item_id","order_id","menu_item_id",
"quantity","unit_price","extended_price"
])

order_items.to_csv(OUTPUT_DIR+"order_items.csv",index=False)

# ------------------------
# shifts
# ------------------------

shifts = []

for i in tqdm(range(NUM_SHIFT_ROWS)):

    emp = random.randint(1,NUM_EMPLOYEES)
    restaurant = employees.loc[
        employees.employee_id==emp,"restaurant_id"
    ].values[0]

    start = START_DATE + timedelta(days=random.randint(0,date_range))

    hours = random.randint(4,10)

    shifts.append([
        i+1,
        emp,
        restaurant,
        start.date(),
        start,
        start+timedelta(hours=hours),
        hours,
        hours,
        employees.loc[employees.employee_id==emp,"job_title"].values[0],
        employees.loc[employees.employee_id==emp,"hourly_rate"].values[0]
    ])

employee_shifts = pd.DataFrame(shifts,columns=[
"shift_id","employee_id","restaurant_id","shift_date",
"shift_start_datetime","shift_end_datetime",
"scheduled_hours","actual_hours","job_title","hourly_rate"
])

employee_shifts.to_csv(OUTPUT_DIR+"employee_shifts.csv",index=False)

# ------------------------
# inventory
# ------------------------

txns = []

for i in tqdm(range(NUM_INVENTORY_TXN)):

    restaurant = random.randint(1,NUM_RESTAURANTS)

    ingredient = random.randint(1,NUM_INGREDIENTS)

    qty = round(random.uniform(1,30),2)

    cost = ingredients.loc[
        ingredients.ingredient_id==ingredient,
        "standard_unit_cost"
    ].values[0]

    txns.append([
        i+1,
        restaurant,
        ingredient,
        START_DATE + timedelta(days=random.randint(0,date_range)),
        random.choice(["RECEIPT","USAGE","WASTE","ADJUSTMENT"]),
        qty,
        cost,
        round(qty*cost,2),
        ""
    ])

inventory = pd.DataFrame(txns,columns=[
"inventory_txn_id","restaurant_id","ingredient_id",
"transaction_datetime","transaction_type","quantity",
"unit_cost","extended_cost","reference_note"
])

inventory.to_csv(OUTPUT_DIR+"inventory_transactions.csv",index=False)

print("All CSV files generated successfully.")

100%|██████████| 200000/200000 [00:29<00:00, 6716.07it/s] 


All CSV files generated successfully.
